In [ ]:
!pip install -q langchain langchain-community langchain-huggingface chromadb bitsandbytes accelerate sentence-transformers pypdf python-dotenv torch transformers

In [ ]:
from google.colab import files
import os

pdf_file = "test.pdf"
if not os.path.exists(pdf_file):
    print("Please upload your PDF file:")
    uploaded = files.upload()
    # Rename uploaded file to test.pdf if needed, or update pdf_file variable

In [ ]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

_embedding_model = None

def get_embedding_model():
    global _embedding_model
    if _embedding_model is None:
        _embedding_model = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
    return _embedding_model

def build_vector_db(pdf_path: str, persist_dir: str):
    if not os.path.exists(pdf_path):
        print(f"Error: PDF file not found at {pdf_path}")
        return None

    emb_model = get_embedding_model()

    print(f"Loading document: {pdf_path}...")
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    print("Splitting document into chunks...")
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100
    )
    chunks = splitter.split_documents(docs)

    if chunks:
        print(f"Storing {len(chunks)} chunks into vector DB at '{persist_dir}'...")
        vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=emb_model,
            persist_directory=persist_dir
        )
        print("Chunks stored successfully!")
        return vectorstore
    else:
        print("No chunks created.")
        return None

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

def get_llm():
    model_id = "microsoft/Phi-3-mini-4k-instruct"

    tokenizer = AutoTokenizer.from_pretrained(model_id, clean_up_tokenization_spaces=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        quantization_config=quantization_config,
        device_map="auto"
    )

    model.generation_config.max_length = None

    pipe = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=300,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id
    )

    return HuggingFacePipeline(pipeline=pipe)

llm = get_llm()
embedding_model = get_embedding_model()

In [ ]:
import time
from langchain_core.messages import SystemMessage, HumanMessage

pdf_file = "test.pdf"
db_directory = "chroma_DB_01"

# Build Database if it does not exist
if not os.path.exists(db_directory):
    print("Building Vector Database from PDF...")
    build_vector_db(pdf_file, db_directory)

# Load the database
vectorstore = Chroma(
    persist_directory=db_directory, embedding_function=embedding_model
)

retriever_response = vectorstore.as_retriever(
    search_type="mmr", search_kwargs={"k": 5, "fetch_k": 10, "lambda_mult": 0.5}
)

print("\nRAG system initialized successfully from disk!")
print("Press '0' to exit.\n")

while True:
    query = input("You : ").strip()
    if query == "0":
        print("Goodbye!")
        break
    if not query:
        continue

    # Retrieve context
    docs = retriever_response.invoke(query)
    context = "\n\n".join([doc.page_content for doc in docs])

    # Create LangChain messages
    messages = [ 
        SystemMessage(
            content="You are a helpful AI assistant. Use only the provided context to answer the question. If the answer is not present in the context, say: 'I could not find the answer in the documnet.'"
        ),
        HumanMessage(
            content=f"Context:\n{context}\n\nQuestion: {query}\n\nRemember: Answer based ONLY on the context. If the answer is not present, say 'I could not find the answer in the documnet.'"
        )
    ]

    # Convert messages to dicts
    formatted_messages = [
        {"role": "system" if isinstance(msg, SystemMessage) else "user", "content": msg.content}
        for msg in messages
    ]
    tokenizer = llm.pipeline.tokenizer
    final_prompt = tokenizer.apply_chat_template(
        formatted_messages, tokenize=False, add_generation_prompt=True
    )

    start_time = time.perf_counter()
    response = llm.invoke(final_prompt)
    end_time = time.perf_counter()

    print(f"\nAI : {response.strip()}")
    print(f"\nResponse Time: {end_time - start_time:.2f} seconds\n")